In [1]:
!pip install opencv-python-headless==4.11.0.86 transformers==4.57.1 Pillow==11.1.0 peft torchvision decord==0.6.0 lmdb==1.7.5 tqdm pillow ultralytics


# Dataset preparation

In [5]:
!python3 dataset.py

subset size: 1000
{'image_id': '2011_001173', 'year': '2012', 'image_set': 'trainval', 'objects': [{'name': 'cat', 'pose': 'Unspecified', 'truncated': '0', 'difficult': '0', 'bbox': {'xmin': 2, 'ymin': 61, 'xmax': 478, 'ymax': 358}}, {'name': 'sofa', 'pose': 'Unspecified', 'truncated': '1', 'difficult': '1', 'bbox': {'xmin': 1, 'ymin': 117, 'xmax': 500, 'ymax': 375}}]}


# YOLO26n & LocateAnything

In [16]:
!python3 inference.py

Streaming output truncated to the last 5000 lines.
image=2011_000220 mIoU=0.9218

0: 448x640 3 persons, 9.5ms
Speed: 1.6ms preprocess, 9.5ms inference, 0.3ms postprocess per image at shape (1, 3, 448, 640)
image=2008_002099 mIoU=0.9150

0: 480x640 1 sheep, 9.8ms
Speed: 1.7ms preprocess, 9.8ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)
image=2010_002294 mIoU=0.0000
Running YOLO26:  26% 259/1000 [00:06<00:14, 49.97it/s]
0: 480x640 1 person, 1 remote, 3 teddy bears, 9.2ms
Speed: 1.7ms preprocess, 9.2ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)
image=2009_002580 mIoU=0.7423

0: 448x640 1 cow, 11.0ms
Speed: 2.6ms preprocess, 11.0ms inference, 0.4ms postprocess per image at shape (1, 3, 448, 640)
image=2009_003089 mIoU=0.0000

0: 480x640 1 person, 9.8ms
Speed: 1.7ms preprocess, 9.8ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)
image=2010_001699 mIoU=0.9584

0: 448x640 1 train, 10.2ms
Speed: 1.6ms preprocess, 10.2ms inference, 

In [13]:

import re

def parse_boxes_org(answer: str, image_width: int, image_height: int) -> list[dict]:
    """Parse model output into pixel-coordinate bounding boxes.

    Coordinates in model output are normalized integers in [0, 1000].
    """
    boxes = []
    for m in re.finditer(r"<box><(\d+)><(\d+)><(\d+)><(\d+)></box>", answer):
        x1, y1, x2, y2 = [int(g) for g in m.groups()]
        boxes.append({
            "x1": x1 / 1000 * image_width,
            "y1": y1 / 1000 * image_height,
            "x2": x2 / 1000 * image_width,
            "y2": y2 / 1000 * image_height,
        })
    return boxes

def parse_boxes(answer: str, image_width: int, image_height: int) -> list[dict]:
    """Parse model output into labeled pixel-coordinate bounding boxes.

    Coordinates in model output are normalized integers in [0, 1000].
    """
    boxes = []
    pattern = re.compile(
        r"<ref>(.*?)</ref>\s*<box>(?:None|<(\d+)><(\d+)><(\d+)><(\d+)>)</box>",
        re.DOTALL,
    )

    for m in pattern.finditer(answer):
        label, x1, y1, x2, y2 = m.groups()
        if None in (x1, y1, x2, y2):
            continue

        boxes.append({
            "class": label.strip().lower(),
            "x1": int(x1) / 1000 * image_width,
            "y1": int(y1) / 1000 * image_height,
            "x2": int(x2) / 1000 * image_width,
            "y2": int(y2) / 1000 * image_height,
        })

    return boxes

print(parse_boxes_org(
    "Answer: <ref>person</ref><box>None</box><ref>aeroplane</ref><box>None</box><ref>car</ref><box>None</box><ref>cat</ref><box><0><165><962><997></box><ref>dog</ref><box>None</box><ref>chair</ref><box><2><315><998><997></box><ref>bottle</ref><box>None</box><|im_end|>",
    375,
    500,
))

print(parse_boxes(
    "Answer: <ref>person</ref><box>None</box><ref>aeroplane</ref><box>None</box><ref>car</ref><box>None</box><ref>cat</ref><box><0><165><962><997></box><ref>dog</ref><box>None</box><ref>chair</ref><box><2><315><998><997></box><ref>bottle</ref><box>None</box><|im_end|>",
    375,
    500,
))

[{'x1': 0.0, 'y1': 82.5, 'x2': 360.75, 'y2': 498.5}, {'x1': 0.75, 'y1': 157.5, 'x2': 374.25, 'y2': 498.5}]
[{'class': 'cat', 'x1': 0.0, 'y1': 82.5, 'x2': 360.75, 'y2': 498.5}, {'class': 'chair', 'x1': 0.75, 'y1': 157.5, 'x2': 374.25, 'y2': 498.5}]


# Yolo26s

In [17]:
!python3 inference.py

2026-06-15 15:27:30.162783: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
YOLO26 device: cuda
Running YOLO26:   0% 0/1000 [00:00<?, ?it/s]
0: 480x640 1 cat, 2 couchs, 62.5ms
Speed: 3.7ms preprocess, 62.5ms inference, 31.9ms postprocess per image at shape (1, 3, 480, 640)
image=2011_001173 mIoU=0.9367
Running YOLO26:   0% 1/1000 [00:00<15:47,  1.05it/s]
0: 448x640 1 train, 3 benchs, 60.3ms
Speed: 2.9ms preprocess, 60.3ms inference, 0.4ms postprocess per image at shape (1, 3, 448, 640)
image=2010_001625 mIoU=0.0000

0: 256x640 1 bird, 59.8ms
Speed: 1.1ms preprocess, 59.8ms inference, 0.6ms postprocess per image at shape (1, 3, 256, 640)
image=2011_003163 mIoU=0.0000
Running YOLO26:   0% 3/1000 [00:01<04:55,  3.37it/s]
0: 640x640 1 dog, 1 couch, 1

# YOLO26m

In [18]:
!python3 inference.py

2026-06-15 15:31:58.554850: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
YOLO26 device: cuda
Running YOLO26:   0% 0/1000 [00:00<?, ?it/s]
0: 480x640 1 cat, 1 couch, 107.1ms
Speed: 4.6ms preprocess, 107.1ms inference, 22.1ms postprocess per image at shape (1, 3, 480, 640)
image=2011_001173 mIoU=0.9268
Running YOLO26:   0% 1/1000 [00:00<16:12,  1.03it/s]
0: 448x640 1 train, 3 benchs, 85.6ms
Speed: 4.0ms preprocess, 85.6ms inference, 0.5ms postprocess per image at shape (1, 3, 448, 640)
image=2010_001625 mIoU=0.0000
Running YOLO26:   0% 2/1000 [00:01<07:39,  2.17it/s]
0: 256x640 1 bird, 84.0ms
Speed: 1.7ms preprocess, 84.0ms inference, 0.7ms postprocess per image at shape (1, 3, 256, 640)
image=2011_003163 mIoU=0.0000

0: 640x640 1 dog, 1 couch, 

# YOLO26l


In [19]:
!python3 inference.py

2026-06-15 15:33:08.676482: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
YOLO26 device: cuda
Running YOLO26:   0% 0/1000 [00:00<?, ?it/s]
0: 480x640 1 cat, 1 couch, 74.0ms
Speed: 3.8ms preprocess, 74.0ms inference, 16.3ms postprocess per image at shape (1, 3, 480, 640)
image=2011_001173 mIoU=0.9276
Running YOLO26:   0% 1/1000 [00:00<12:05,  1.38it/s]
0: 448x640 1 train, 2 benchs, 62.5ms
Speed: 3.8ms preprocess, 62.5ms inference, 0.5ms postprocess per image at shape (1, 3, 448, 640)
image=2010_001625 mIoU=0.0000

0: 256x640 1 bird, 60.8ms
Speed: 1.0ms preprocess, 60.8ms inference, 0.5ms postprocess per image at shape (1, 3, 256, 640)
image=2011_003163 mIoU=0.0000
Running YOLO26:   0% 3/1000 [00:00<04:00,  4.14it/s]
0: 640x640 1 dog, 1 couch, 48

# YOLO26x

In [20]:
!python3 inference.py

2026-06-15 15:36:54.900130: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
YOLO26 device: cuda
Running YOLO26:   0% 0/1000 [00:00<?, ?it/s]
0: 480x640 1 cat, 1 couch, 120.6ms
Speed: 6.0ms preprocess, 120.6ms inference, 25.0ms postprocess per image at shape (1, 3, 480, 640)
image=2011_001173 mIoU=0.9333
Running YOLO26:   0% 1/1000 [00:01<17:21,  1.04s/it]
0: 448x640 1 person, 1 train, 3 benchs, 93.9ms
Speed: 4.2ms preprocess, 93.9ms inference, 0.7ms postprocess per image at shape (1, 3, 448, 640)
image=2010_001625 mIoU=0.0000
Running YOLO26:   0% 2/1000 [00:01<08:13,  2.02it/s]
0: 256x640 1 bird, 86.2ms
Speed: 2.1ms preprocess, 86.2ms inference, 0.7ms postprocess per image at shape (1, 3, 256, 640)
image=2011_003163 mIoU=0.0000

0: 640x640 1 dog,

# Visualize

In [3]:
!python3 visualize.py

2026-06-15 16:12:48.395625: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
YOLO26 device: cuda
LocateAnything device: cuda
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
`torch_dtype` is depr

In [4]:
import os
from google.colab import files

# Zip the 'figure' directory
!zip -r figures.zip figure

# Download the zip file
files.download('figures.zip')

  adding: figure/ (stored 0%)
  adding: figure/visualize_0009_2010_003594.png (deflated 15%)
  adding: figure/visualize_0006_2008_002448.png (deflated 9%)
  adding: figure/visualize_0002_2011_003163.png (deflated 15%)
  adding: figure/visualize_0004_2009_000686.png (deflated 15%)
  adding: figure/visualize_0008_2008_008434.png (deflated 2%)
  adding: figure/visualize_0001_2010_001625.png (deflated 15%)
  adding: figure/visualize_0000_2011_001173.png (deflated 15%)
  adding: figure/visualize_0005_2008_006502.png (deflated 14%)
  adding: figure/visualize_0007_2008_002172.png (deflated 14%)
  adding: figure/visualize_0003_2009_000416.png (deflated 1%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>